<a href="https://colab.research.google.com/github/eduardokern/ML/blob/face_detection/notebooks/face_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Criando um sistema de reconhecimento facial do zero</h1>

O objetivo principal deste projeto é trabalhar com as bibliotecas e frameworks estudados e analisados em nossas aulas. Neste sentido, a proposta padrão envolve um sistema de detecção e reconhecimento de faces, utilizando o framework TensorFlow em conjuntos com as bibliotecas que o projetista julgue necessárias, de forma ilimitada.  

Por meio da Figura 1 é possível visualizar o resultado esperado para o modelo proposto, devendo detectar e reconhecer mais de uma face ao mesmo tempo.  
Para isso você deve:

1. Utilizar uma rede de detecção treinada para detectar faces.
2. Utilizar uma rede de classificação para classificar a face detectada.

![Figura 1: Detecção e reconhecimento facial.](https://drive.google.com/uc?export=view&id=1nFCZd-FR0jIYAvDbDbVpDt6ansjEkLIA)

Para realizar este projeto, você pode utilizar os seguintes trabalhos de referência:
Detecção Facial:
https://colab.research.google.com/drive/1QnC7lV7oVFk5OZCm75fqbLAfD9qBy9bw?usp=sharing

Detecção e classificação de objetos:  
https://colab.research.google.com/drive/1xdjyBiY75MAVRSjgmiqI7pbRLn58VrbE?usp=sharing

In [1]:
# Mount google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import tensorflow as tf
import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense

In [ ]:
class FaceDetection:
  def __init__(self, input_shape):
    self._detection_model(input_shape)
    self._train_detection()

  def _detection_model(self, input_shape):
    inputs = Input(shape=input_shape)

    # Convolutional layers
    x = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = MaxPooling2D((2, 2))(x)
    x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)
    x = Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)

    # Flatten and Dense layers
    x = Flatten()(x)
    x = Dense(256, activation='relu')(x)
    x = Dense(128, activation='relu')(x)

    # Output layer for bounding box regression
    # 4 values: x, y, width, height
    outputs = Dense(4, activation='linear')(x)

    self._detection_model = Model(inputs, outputs)

  def _train_detection(self, train_data, val_data, epochs, batch_size):
    self._detection_model.compile(optimizer='adam', loss='mse', metrics=['accuracy'])
    self._detection_model.fit(
        train_data[0],
        train_data[1],
        validation_data=val_data,
        epochs=epochs,
        batch_size=batch_size)

  def train_classification(self, num_classes, train_data, val_data, epochs, batch_size):
    vgg = keras.applications.VGG16(weights='imagenet', include_top=True)
    inp = vgg.input

    # make a new softmax layer with num_classes neurons
    new_classification_layer = Dense(num_classes, activation='softmax')

    # connect our new layer to the second to last layer in VGG, and make a reference to it
    out = new_classification_layer(vgg.layers[-2].output)

    # create a new network between inp and out
    self._classification_model = Model(inp, out)

    # make all layers untrainable by freezing weights (except for last layer)
    for l, layer in enumerate(self._classification_model.layers[:-1]):
        layer.trainable = False

    # ensure the last layer is trainable/not frozen
    for l, layer in enumerate(self._classification_model.layers[-1:]):
        layer.trainable = True

    self._classification_model.compile(
        loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    self._detection_model.fit(
        train_data[0],
        train_data[1],
        validation_data=val_data,
        epochs=epochs,
        batch_size=batch_size)

